# Continuous Batching

**Stage 5: Inference Optimization - Notebook 59**

This notebook explores continuous batching (also called in-flight batching or iteration-level scheduling), a critical technique for maximizing LLM serving throughput. We'll cover:
- Dynamic batching concept
- Orca scheduling algorithm (Microsoft, 2022)
- Request queueing strategies
- Iteration-level scheduling
- Throughput optimization analysis
- Implementing continuous batching
- Comparison with static batching
- Production considerations

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
import time
from collections import deque
import random

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. The Problem with Static Batching

### Historical Context

**Traditional Batching (Pre-2022)**:
```python
# Static batching
batch = collect_requests(batch_size=32)
for step in range(max_len):
    outputs = model(batch)  # All 32 requests
# All requests finish together
```

**Problems**:
1. **Padding Waste**: Short sequences padded to longest in batch
2. **Idle Time**: Must wait for slowest request to finish
3. **Variable Latency**: Short requests delayed by long ones
4. **Underutilization**: GPU idle when waiting for batch to fill

**Example**:
```
Request A: 10 tokens needed
Request B: 100 tokens needed
Request C: 20 tokens needed

Static batching:
- All process together for 100 steps
- A and C wait 80-90 steps doing nothing!
- GPU computes padding (wasted)
```

**The Innovation (Orca, 2022)**:
- Add/remove requests at EVERY iteration
- No waiting for batch to complete
- No padding waste
- Continuous GPU utilization

In [ ]:
# Visualize the problem
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Static batching
ax = axes[0]
requests_static = [
    ('Request A', 10, '#3498db'),
    ('Request B', 100, '#e74c3c'),
    ('Request C', 20, '#2ecc71'),
    ('Request D', 15, '#f39c12'),
]

y_pos = 0
for name, length, color in requests_static:
    # Actual computation
    ax.barh(y_pos, length, left=0, height=0.8, color=color, alpha=0.7, label=f"{name} (active)")
    # Wasted time (waiting for longest)
    if length < 100:
        ax.barh(y_pos, 100-length, left=length, height=0.8, color='gray', alpha=0.3, 
               edgecolor='red', linewidth=2, linestyle='--')
    ax.text(length/2, y_pos, name, ha='center', va='center', fontsize=10, fontweight='bold')
    y_pos += 1

ax.set_xlim(0, 110)
ax.set_ylim(-0.5, len(requests_static))
ax.set_xlabel('Iteration (tokens)', fontsize=12)
ax.set_ylabel('Request', fontsize=12)
ax.set_title('Static Batching: All requests wait for slowest to finish\n' + 
             'Gray areas = WASTED computation (padding)', 
             fontsize=13, fontweight='bold', color='red')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

# Add annotation
ax.annotate('', xy=(100, 3.5), xytext=(10, 3.5),
           arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(55, 3.7, '90 wasted iterations!', ha='center', fontsize=11, 
       fontweight='bold', color='red')

# Continuous batching
ax = axes[1]

# Timeline showing dynamic batching
timeline = [
    # (start, end, request, color)
    (0, 10, 'A', '#3498db'),
    (0, 20, 'C', '#2ecc71'),
    (0, 15, 'D', '#f39c12'),
    (0, 100, 'B', '#e74c3c'),
    (11, 30, 'E', '#9b59b6'),  # New request added when A finishes
    (21, 45, 'F', '#1abc9c'),  # New request added when C finishes
    (31, 50, 'G', '#e67e22'),  # New request added when E finishes
]

# Group by active slots
max_slots = 4
slot_assignments = {}
for start, end, name, color in timeline:
    # Find available slot
    for slot in range(max_slots):
        conflict = False
        for s, e, n, c in slot_assignments.get(slot, []):
            if not (end <= s or start >= e):  # Overlap check
                conflict = True
                break
        if not conflict:
            if slot not in slot_assignments:
                slot_assignments[slot] = []
            slot_assignments[slot].append((start, end, name, color))
            break

# Draw timeline
for slot, events in slot_assignments.items():
    for start, end, name, color in events:
        ax.barh(slot, end-start, left=start, height=0.8, color=color, alpha=0.7)
        ax.text((start+end)/2, slot, f'Req {name}', ha='center', va='center', 
               fontsize=9, fontweight='bold')

ax.set_xlim(0, 110)
ax.set_ylim(-0.5, max_slots)
ax.set_xlabel('Iteration (tokens)', fontsize=12)
ax.set_ylabel('Batch Slot', fontsize=12)
ax.set_title('Continuous Batching: Requests added/removed dynamically\n' +
             'NO wasted computation, GPU always busy',
             fontsize=13, fontweight='bold', color='green')
ax.set_yticks(range(max_slots))
ax.set_yticklabels([f'Slot {i}' for i in range(max_slots)])
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. Static batching: 100 iterations for 4 requests (145 total tokens)")
print("2. Continuous batching: 100 iterations for 7 requests (240 total tokens)")
print("3. Throughput improvement: 1.66x (240/145)")
print("4. Latency improvement: Short requests don't wait for long ones")

## 2. Orca Scheduling Algorithm

### Historical Context

**Paper**: "Orca: A Distributed Serving System for Transformer-Based Generative Models" (Yu et al., Microsoft, 2022)

**Key Innovation**: Iteration-level scheduling
- Traditional: Batch-level scheduling (entire batch together)
- Orca: Iteration-level scheduling (modify batch at each step)

**Algorithm**:
```python
running_requests = []
waiting_queue = Queue()

while True:
    # 1. Remove finished requests
    running_requests = [r for r in running_requests if not r.finished()]
    
    # 2. Add new requests from queue (if space)
    while len(running_requests) < max_batch_size and not waiting_queue.empty():
        running_requests.append(waiting_queue.pop())
    
    # 3. Execute one iteration for all running requests
    outputs = model.forward(running_requests)
    
    # 4. Update states
    for request, output in zip(running_requests, outputs):
        request.append_token(output)
```

**Benefits**:
- 2-3x throughput improvement
- Better latency for short requests
- No padding waste
- Continuous GPU utilization

**Impact**:
- Adopted by vLLM, TensorRT-LLM, TGI
- Now standard in all modern LLM serving systems
- Renamed "continuous batching" or "in-flight batching"

In [ ]:
@dataclass
class Request:
    """Represents a single generation request."""
    id: int
    prompt_tokens: List[int]
    max_tokens: int
    generated_tokens: List[int]
    arrival_time: float
    start_time: Optional[float] = None
    finish_time: Optional[float] = None
    
    def __post_init__(self):
        if not self.generated_tokens:
            self.generated_tokens = []
    
    def is_finished(self) -> bool:
        """Check if request is done generating."""
        return len(self.generated_tokens) >= self.max_tokens
    
    def current_length(self) -> int:
        """Total length including prompt."""
        return len(self.prompt_tokens) + len(self.generated_tokens)
    
    def waiting_time(self) -> float:
        """Time spent waiting in queue."""
        if self.start_time is None:
            return time.time() - self.arrival_time
        return self.start_time - self.arrival_time
    
    def execution_time(self) -> Optional[float]:
        """Time spent executing."""
        if self.finish_time is None or self.start_time is None:
            return None
        return self.finish_time - self.start_time


class OrcaScheduler:
    """
    Orca-style continuous batching scheduler.
    
    Key features:
    - Iteration-level scheduling
    - Dynamic batch modification
    - FIFO queue with max batch size
    """
    def __init__(self, max_batch_size: int, max_queue_size: int = 1000):
        self.max_batch_size = max_batch_size
        self.max_queue_size = max_queue_size
        
        self.running_requests: List[Request] = []
        self.waiting_queue: deque = deque(maxlen=max_queue_size)
        self.completed_requests: List[Request] = []
        
        self.current_iteration = 0
        self.stats = {
            'total_requests': 0,
            'completed_requests': 0,
            'rejected_requests': 0,
            'total_iterations': 0,
            'total_tokens_generated': 0,
        }
    
    def add_request(self, request: Request) -> bool:
        """Add a new request to the queue."""
        if len(self.waiting_queue) >= self.max_queue_size:
            self.stats['rejected_requests'] += 1
            return False
        
        self.waiting_queue.append(request)
        self.stats['total_requests'] += 1
        return True
    
    def schedule_iteration(self) -> List[Request]:
        """
        Schedule one iteration (Orca algorithm).
        
        Returns:
            List of requests to process in this iteration
        """
        # Step 1: Remove finished requests
        finished = [r for r in self.running_requests if r.is_finished()]
        for request in finished:
            request.finish_time = time.time()
            self.completed_requests.append(request)
            self.stats['completed_requests'] += 1
            self.stats['total_tokens_generated'] += len(request.generated_tokens)
        
        self.running_requests = [r for r in self.running_requests if not r.is_finished()]
        
        # Step 2: Add new requests from queue (fill up to max_batch_size)
        while len(self.running_requests) < self.max_batch_size and self.waiting_queue:
            request = self.waiting_queue.popleft()
            request.start_time = time.time()
            self.running_requests.append(request)
        
        # Step 3: Return current batch
        self.current_iteration += 1
        self.stats['total_iterations'] += 1
        
        return self.running_requests
    
    def get_stats(self) -> Dict:
        """Get scheduler statistics."""
        stats = self.stats.copy()
        stats['running_requests'] = len(self.running_requests)
        stats['waiting_requests'] = len(self.waiting_queue)
        
        if self.stats['completed_requests'] > 0:
            avg_wait = sum(r.waiting_time() for r in self.completed_requests) / len(self.completed_requests)
            avg_exec = sum(r.execution_time() for r in self.completed_requests if r.execution_time()) / len(self.completed_requests)
            stats['avg_waiting_time'] = avg_wait
            stats['avg_execution_time'] = avg_exec
        
        if self.stats['total_iterations'] > 0:
            stats['avg_batch_size'] = len(self.running_requests)
            stats['throughput_tokens_per_iter'] = self.stats['total_tokens_generated'] / self.stats['total_iterations']
        
        return stats

print("Orca Scheduler Implemented")
print("="*70)
print("\nKey Features:")
print("1. Iteration-level scheduling (not batch-level)")
print("2. Dynamic batch modification at each step")
print("3. FIFO queue with max batch size constraint")
print("4. Automatic removal of finished requests")
print("5. Continuous addition of new requests")

## 3. Simulation: Static vs Continuous Batching

In [ ]:
def simulate_static_batching(requests: List[Request], batch_size: int) -> Dict:
    """
    Simulate static batching.
    Process requests in fixed batches.
    """
    results = {
        'total_iterations': 0,
        'total_tokens': 0,
        'total_time': 0,
        'request_latencies': [],
    }
    
    # Process in batches
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        if not batch:
            break
        
        # All requests in batch process together
        max_length = max(r.max_tokens for r in batch)
        
        for iteration in range(max_length):
            results['total_iterations'] += 1
            # Count actual tokens generated (not padding)
            for request in batch:
                if len(request.generated_tokens) < request.max_tokens:
                    request.generated_tokens.append(0)  # Dummy token
                    results['total_tokens'] += 1
        
        # All requests finish at same time (end of batch)
        batch_time = max_length
        for j, request in enumerate(batch):
            # Latency includes waiting for previous batches + this batch
            latency = results['total_time'] + batch_time
            results['request_latencies'].append(latency)
        
        results['total_time'] += batch_time
    
    return results


def simulate_continuous_batching(requests: List[Request], max_batch_size: int) -> Dict:
    """
    Simulate continuous batching (Orca-style).
    """
    scheduler = OrcaScheduler(max_batch_size=max_batch_size)
    
    # Add all requests to queue
    for request in requests:
        scheduler.add_request(request)
    
    iteration = 0
    request_latencies = {}
    
    # Run until all requests complete
    while scheduler.running_requests or scheduler.waiting_queue:
        batch = scheduler.schedule_iteration()
        
        if not batch:
            break
        
        # Simulate one iteration
        for request in batch:
            if not request.is_finished():
                request.generated_tokens.append(0)  # Dummy token
                
                # Record latency when request finishes
                if request.is_finished() and request.id not in request_latencies:
                    request_latencies[request.id] = iteration + 1
        
        iteration += 1
    
    results = {
        'total_iterations': iteration,
        'total_tokens': scheduler.stats['total_tokens_generated'],
        'total_time': iteration,
        'request_latencies': [request_latencies[i] for i in range(len(requests))],
    }
    
    return results


# Generate test requests with varying lengths
random.seed(42)
num_requests = 50
test_requests_static = [
    Request(
        id=i,
        prompt_tokens=[0] * 10,
        max_tokens=random.randint(10, 100),  # Variable output length
        generated_tokens=[],
        arrival_time=0.0
    )
    for i in range(num_requests)
]

# Copy for continuous batching
test_requests_continuous = [
    Request(
        id=r.id,
        prompt_tokens=r.prompt_tokens.copy(),
        max_tokens=r.max_tokens,
        generated_tokens=[],
        arrival_time=r.arrival_time
    )
    for r in test_requests_static
]

print("Running Simulations...")
print("="*70)

batch_size = 8

# Static batching
print(f"\nStatic Batching (batch_size={batch_size}):")
static_results = simulate_static_batching(test_requests_static, batch_size)
print(f"  Total iterations: {static_results['total_iterations']}")
print(f"  Total tokens: {static_results['total_tokens']}")
print(f"  Total time: {static_results['total_time']}")
print(f"  Avg latency: {np.mean(static_results['request_latencies']):.1f}")
print(f"  P95 latency: {np.percentile(static_results['request_latencies'], 95):.1f}")

# Continuous batching
print(f"\nContinuous Batching (max_batch_size={batch_size}):")
continuous_results = simulate_continuous_batching(test_requests_continuous, batch_size)
print(f"  Total iterations: {continuous_results['total_iterations']}")
print(f"  Total tokens: {continuous_results['total_tokens']}")
print(f"  Total time: {continuous_results['total_time']}")
print(f"  Avg latency: {np.mean(continuous_results['request_latencies']):.1f}")
print(f"  P95 latency: {np.percentile(continuous_results['request_latencies'], 95):.1f}")

# Improvements
print(f"\nImprovements:")
print(f"  Time reduction: {static_results['total_time'] / continuous_results['total_time']:.2f}x")
print(f"  Throughput increase: {continuous_results['total_tokens'] / continuous_results['total_time']:.1f} vs "
      f"{static_results['total_tokens'] / static_results['total_time']:.1f} tokens/iter")
print(f"  Avg latency improvement: {np.mean(static_results['request_latencies']) / np.mean(continuous_results['request_latencies']):.2f}x")
print(f"  P95 latency improvement: {np.percentile(static_results['request_latencies'], 95) / np.percentile(continuous_results['request_latencies'], 95):.2f}x")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Latency distribution
ax = axes[0, 0]
ax.hist(static_results['request_latencies'], bins=20, alpha=0.6, label='Static', color='#e74c3c')
ax.hist(continuous_results['request_latencies'], bins=20, alpha=0.6, label='Continuous', color='#2ecc71')
ax.set_xlabel('Latency (iterations)', fontsize=12)
ax.set_ylabel('Number of Requests', fontsize=12)
ax.set_title('Latency Distribution', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Latency by request (sorted)
ax = axes[0, 1]
sorted_static = sorted(static_results['request_latencies'])
sorted_continuous = sorted(continuous_results['request_latencies'])
x = np.arange(len(sorted_static))

ax.plot(x, sorted_static, label='Static', color='#e74c3c', linewidth=2)
ax.plot(x, sorted_continuous, label='Continuous', color='#2ecc71', linewidth=2)
ax.set_xlabel('Request (sorted by latency)', fontsize=12)
ax.set_ylabel('Latency (iterations)', fontsize=12)
ax.set_title('Latency per Request (Sorted)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Timeline comparison
ax = axes[1, 0]
metrics = ['Total\nTime', 'Avg\nLatency', 'P95\nLatency']
static_vals = [
    static_results['total_time'],
    np.mean(static_results['request_latencies']),
    np.percentile(static_results['request_latencies'], 95)
]
continuous_vals = [
    continuous_results['total_time'],
    np.mean(continuous_results['request_latencies']),
    np.percentile(continuous_results['request_latencies'], 95)
]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, static_vals, width, label='Static', color='#e74c3c', alpha=0.7)
bars2 = ax.bar(x + width/2, continuous_vals, width, label='Continuous', color='#2ecc71', alpha=0.7)

ax.set_ylabel('Iterations', fontsize=12)
ax.set_title('Performance Metrics', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add improvement labels
for i in range(len(metrics)):
    improvement = static_vals[i] / continuous_vals[i]
    ax.text(i, max(static_vals[i], continuous_vals[i]) * 1.05,
           f'{improvement:.2f}x', ha='center', fontsize=10, fontweight='bold', color='green')

# Throughput over time
ax = axes[1, 1]
# Simplified throughput calculation
static_throughput = static_results['total_tokens'] / static_results['total_time']
continuous_throughput = continuous_results['total_tokens'] / continuous_results['total_time']

methods = ['Static\nBatching', 'Continuous\nBatching']
throughputs = [static_throughput, continuous_throughput]
colors_tput = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, throughputs, color=colors_tput, alpha=0.7)
ax.set_ylabel('Throughput (tokens/iteration)', fontsize=12)
ax.set_title('Average Throughput', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, throughputs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{val:.2f}',
           ha='center', va='bottom', fontsize=11, fontweight='bold')

improvement = continuous_throughput / static_throughput
ax.text(0.5, max(throughputs) * 0.95,
       f'{improvement:.2f}x improvement',
       ha='center', fontsize=12, fontweight='bold', color='green',
       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))

plt.tight_layout()
plt.show()

## 4. Advanced Scheduling Policies

Beyond basic FIFO, various scheduling policies can optimize for different objectives.

In [ ]:
print("Advanced Scheduling Policies")
print("="*80)

scheduling_policies = """
1. FIFO (First-In-First-Out):
   - Default in most systems
   - Fair: requests served in arrival order
   - Simple to implement
   - May lead to head-of-line blocking
   
   Use when: Fairness is priority


2. Shortest Job First (SJF):
   - Prioritize requests with fewer expected tokens
   - Minimizes average latency
   - Can starve long requests
   - Requires length estimation
   
   Implementation:
   queue.sort(key=lambda r: r.estimated_tokens)
   
   Use when: Minimize average latency


3. Priority-Based:
   - Assign priority to requests (e.g., paid users)
   - Higher priority = earlier execution
   - Can implement priority levels
   - May require fairness mechanisms
   
   Implementation:
   queue.sort(key=lambda r: (-r.priority, r.arrival_time))
   
   Use when: Different SLAs for different users


4. Preemptive Scheduling:
   - Can pause long-running requests
   - Save KV cache, resume later
   - Reduces head-of-line blocking
   - More complex implementation
   
   vLLM supports this via preemption!
   
   Use when: Need low latency SLAs


5. Weighted Fair Queueing:
   - Balance between throughput and fairness
   - Each request gets fair share of resources
   - Prevents starvation
   - Complex to implement
   
   Use when: Need fairness across diverse workloads


6. Deadline-Aware:
   - Requests have explicit deadlines
   - Schedule to meet deadlines (EDF - Earliest Deadline First)
   - May drop requests that miss deadline
   
   Implementation:
   queue.sort(key=lambda r: r.deadline)
   
   Use when: Time-critical applications


7. Multi-Level Feedback Queue:
   - Multiple queues with different priorities
   - New requests start in high priority
   - Move to lower priority if taking too long
   - Balances latency and throughput
   
   Use when: Mixed workload (short + long requests)


8. Speculation-Aware:
   - Consider draft model acceptance rate
   - Prioritize requests likely to accept drafts
   - Maximize speculative decoding benefits
   
   Use when: Using speculative decoding
"""

print(scheduling_policies)

# Comparison table
import pandas as pd

comparison = {
    'Policy': ['FIFO', 'SJF', 'Priority', 'Preemptive', 'WFQ', 'Deadline', 'MLFQ'],
    'Fairness': ['High', 'Low', 'Medium', 'Medium', 'High', 'Medium', 'High'],
    'Avg Latency': ['Medium', 'Best', 'Varies', 'Good', 'Good', 'Good', 'Good'],
    'Complexity': ['Low', 'Low', 'Low', 'High', 'High', 'Medium', 'High'],
    'Starvation Risk': ['No', 'Yes', 'Yes', 'No', 'No', 'Possible', 'No'],
    'Best For': [
        'General purpose',
        'Minimize avg latency',
        'Multi-tenant',
        'Low latency SLAs',
        'Fair sharing',
        'Time-critical',
        'Mixed workloads'
    ]
}

df = pd.DataFrame(comparison)
print("\n\nScheduling Policy Comparison:")
print("="*80)
print(df.to_string(index=False))

## 5. Production Considerations

In [ ]:
print("Continuous Batching - Production Best Practices")
print("="*80)

best_practices = """
1. BATCH SIZE CONFIGURATION:

   Max Batch Size:
   - Set based on GPU memory
   - Too small: underutilization
   - Too large: OOM errors
   - Start with: memory_gb / (model_size_gb + kv_cache_per_request_gb)
   
   Example (A100 80GB, Llama 2 13B):
   - Model: 26 GB
   - KV cache per request (2K context): ~200 MB
   - Available for KV: 54 GB
   - Max batch size: 54 / 0.2 ≈ 270 requests
   - Recommended: 200 (leave margin)


2. QUEUE MANAGEMENT:

   Queue Size:
   ✓ Set max queue size (prevent unbounded growth)
   ✓ Monitor queue length (alert if > threshold)
   ✓ Reject when queue full (fail fast)
   
   Queue Timeout:
   ✓ Set max waiting time
   ✓ Cancel requests that wait too long
   ✓ Return timeout error to client
   
   Example:
   max_queue_size = max_batch_size * 10
   max_wait_time = 30 seconds


3. MEMORY MANAGEMENT:

   KV Cache:
   ✓ Use PagedAttention (avoid fragmentation)
   ✓ Monitor cache utilization
   ✓ Implement eviction policy if needed
   
   OOM Handling:
   ✓ Catch OOM exceptions
   ✓ Reduce batch size dynamically
   ✓ Preempt low-priority requests
   ✓ Log and alert


4. MONITORING METRICS:

   Critical:
   - Queue length (current, max, avg)
   - Batch size (current, avg)
   - Waiting time (P50, P95, P99)
   - Execution time per request
   - Throughput (tokens/sec, requests/sec)
   - GPU utilization
   - Memory usage
   
   Alerts:
   - Queue length > 80% of max
   - Waiting time > SLA threshold
   - GPU utilization < 80% (underutilized)
   - Memory usage > 95%
   - Request rejection rate > 1%


5. OPTIMIZATION CHECKLIST:

   ✓ Profile batch size vs throughput
   ✓ Tune based on request length distribution
   ✓ Use preemption for latency-sensitive requests
   ✓ Implement priority levels if needed
   ✓ Monitor and adjust max_batch_size
   ✓ Set appropriate timeouts
   ✓ Enable prefix caching for common prompts
   ✓ Use continuous batching in all frameworks!


6. LOAD TESTING:

   Test scenarios:
   a) Sustained load (constant arrival rate)
   b) Burst traffic (spike to 10x normal)
   c) Variable length (mix of short/long)
   d) Cold start (from empty queue)
   
   Measure:
   - Maximum throughput
   - Latency at different loads
   - Queue behavior under stress
   - Memory pressure
   - Recovery time after burst


7. FRAMEWORK-SPECIFIC TIPS:

   vLLM:
   --max-num-seqs 256
   --max-num-batched-tokens 8192
   --enable-prefix-caching
   
   TensorRT-LLM:
   --max_batch_size 128
   --max_num_tokens 8192
   --enable_trt_overlap
   
   Text Generation Inference:
   --max-batch-prefill-tokens 4096
   --max-batch-total-tokens 8192


8. COMMON PITFALLS:

   ✗ Not monitoring queue length
   ✗ Setting max_batch_size too high (OOM)
   ✗ No timeout for waiting requests
   ✗ Ignoring request length distribution
   ✗ Not using PagedAttention (memory waste)
   ✗ No alerts for queue buildup
   ✗ Not load testing before production
   ✗ Assuming FIFO is always best
"""

print(best_practices)

## Summary

### Key Takeaways

1. **The Problem**:
   - Static batching: All requests wait for slowest to finish
   - Padding waste: GPU computes on padding tokens
   - Poor utilization: GPU idle between batches
   - Variable latency: Short requests blocked by long ones

2. **The Solution - Continuous Batching**:
   - Iteration-level scheduling (not batch-level)
   - Add requests as slots free up
   - Remove requests as they finish
   - No padding waste
   - Continuous GPU utilization

3. **Performance Gains**:
   - **Throughput**: 2-3x improvement typical
   - **Latency**: 2-5x better for short requests
   - **Utilization**: Near 100% GPU utilization
   - **Efficiency**: No wasted computation on padding

4. **Orca Algorithm** (Microsoft, 2022):
   ```python
   while True:
       # Remove finished
       batch = [r for r in batch if not r.done()]
       # Add new (up to max)
       batch += queue.pop_until(max_batch_size)
       # Execute one iteration
       model(batch)
   ```

5. **Scheduling Policies**:
   - **FIFO**: Default, fair, simple
   - **SJF**: Best average latency, may starve
   - **Priority**: For multi-tenant, SLAs
   - **Preemptive**: Best for latency SLAs
   - **MLFQ**: Best for mixed workloads

6. **Production Best Practices**:
   - Set max_batch_size based on memory
   - Monitor queue length continuously
   - Set timeouts for waiting requests
   - Use PagedAttention to avoid fragmentation
   - Profile with realistic workload
   - Choose appropriate scheduling policy

7. **Framework Support**:
   - **vLLM**: Native continuous batching (recommended)
   - **TensorRT-LLM**: In-flight batching
   - **Text Generation Inference**: Dynamic batching
   - **HuggingFace Transformers**: No built-in support

### Historical Impact

- **2022**: Orca paper introduces iteration-level scheduling
- **2023**: Adopted by vLLM, TensorRT-LLM
- **2023-2024**: Became standard in all production frameworks
- **Impact**: Enabled 2-3x throughput improvement, critical for viable LLM serving

### Why It Matters

Continuous batching is **essential** for production LLM serving:

1. **Economics**: 2-3x throughput = 2-3x less hardware needed
2. **Latency**: Better user experience for interactive applications
3. **Efficiency**: No wasted computation (cost savings)
4. **Utilization**: Keep expensive GPUs busy

**Without continuous batching, modern LLM serving would not be economically viable at scale.**

### Recommendation

**Always use continuous batching for production LLM serving:**
- Use vLLM (easiest, best default)
- Or TensorRT-LLM (best performance on NVIDIA)
- Never use static batching in production
- Monitor queue metrics continuously
- Choose scheduling policy based on workload

### Stage 5 Complete!

You've completed the Inference Optimization stage. You now understand:
- KV cache optimization (GQA, PagedAttention)
- Speculative decoding (2-3x speedup)
- vLLM serving (production standard)
- TensorRT-LLM (maximum performance)
- Continuous batching (2-3x throughput)

These techniques combined enable practical, cost-effective LLM serving at scale.